In [ ]:
'''This part contains all the code for my content and style reconstruction experiments'''

In [ ]:
'''This is the initial approach I used to visualize content feature maps at different processing stages of CNN.'''
def visualize_feature_maps (features):
  for layer_name, activation in features.items():
    print(activation.shape)
    img_data = activation.squeeze(0).cpu().detach().numpy()

    # If the feature map has multiple channels, display only the first channel
    if img_data.ndim == 3: # (C, H, W) format
        display_img = img_data[0, :, :]
    elif img_data.ndim == 2: # (H, W) format
        display_img = img_data
    else:
        print("Skipping.")
        continue

    plt.imshow(display_img, cmap = "gray")
    plt.title(layer_name)
    plt.show()

In [ ]:
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)
normalize = T.Normalize(mean=MEAN, std=STD)

'''For style reconstruction, this is my first experiment. This function aims to minimize the mean squared error (MSE) between a random style image and a random white
noise image's feature maps'''

def reconstruct_features(target_image, layer_name, num_steps=10000, lr=0.1):

  layers = {'3': 'relu_1_2',
            '8': 'relu_2_2',
            '15': 'relu_3_3',
            '22': 'relu_4_3'}

  vgg = vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features
  for param in vgg.parameters():
    param.requires_grad = False

  loss_network = create_feature_extractor(vgg, layers)
  loss_network = loss_network.to(device)

  target_features = loss_network(normalize(target_image))[layer_name]
  target_features = target_features.to(device)

  reconstructed_image = torch.randn_like(target_image, requires_grad=True)
  reconstructed_image = reconstructed_image.to(device)
  optimizer = torch.optim.Adam([reconstructed_image], lr=lr)

  for step in range(num_steps):
    reconstructed_features = loss_network(normalize(reconstructed_image))[layer_name]
    reconstructed_features = reconstructed_features.to(device)
    loss = torch.mean((reconstructed_features - target_features) ** 2)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

  reconstructed_image = reconstructed_image.detach().clamp(0, 1)
  display(to_pil_image(reconstructed_image.squeeze(0)))

  return


In [ ]:
from torch.nn.functional import mse_loss
'''This is my second style reconstruction experiment. I attempted to improve my first approach (the cell before) by matching the MSE of the Gram Matrices computed
across a random style image and a random white noise image's extracted stylistic feature maps. I believe that since computing the Gram Matrices requires multiplication
of different channels representing stylistic features across the spatial dimensions of a feature map, matching the MSE of those would result in better reconstruction of
stylistic features. '''

def reconstruct_style_features (target_image, layer_name, num_steps = 10000, lr = 0.1):
  layers = {'3': 'relu_1_2',
            '8': 'relu_2_2',
            '15': 'relu_3_3',
            '22': 'relu_4_3'}

  vgg = vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features
  for param in vgg.parameters():
    param.requires_grad = False

  loss_network = create_feature_extractor(vgg, layers)
  loss_network = loss_network.to(device)

  target_features = loss_network(normalize(target_image))[layer_name]
  target_features = target_features.to(device)

  reconstructed_image = torch.randn_like(target_image, requires_grad=True)
  reconstructed_image = reconstructed_image.to(device)
  optimizer = torch.optim.Adam([reconstructed_image], lr=lr)

  for step in range(num_steps):

    if (step%100==0): print(f"Step {step}")
    reconstructed_features = loss_network(normalize(reconstructed_image))[layer_name]
    reconstructed_features = reconstructed_features.to(device)
    loss = mse_loss(gram(reconstructed_features), gram(target_features))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

  reconstructed_image = reconstructed_image.detach().clamp(0, 1)
  display(to_pil_image(reconstructed_image.squeeze(0)))

  return



In [ ]:

'''Version 3 of style reconstruction experiment. I adjusted my code for the second approach to try to visualize the reconstructed stylistic features that my second experiment
in the paper (calculating average Gram Matrices across each set of style-associated images for 3 styles being modelled) tries to capture. '''
def style_gram_features (style_id, layer_name, num_steps = 10000, lr = 0.1):
  layers = {'3': 'relu_1_2',
            '8': 'relu_2_2',
            '15': 'relu_3_3',
            '22': 'relu_4_3'}

  vgg = vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features
  for param in vgg.parameters():
    param.requires_grad = False

  loss_network = create_feature_extractor(vgg, layers)
  loss_network = loss_network.to(device)

  style_summary = precomputed_style_grams[style_id][layer_name]

  # Initialize with a default size since no target_image is provided to this function
  reconstructed_image = torch.randn(1, 3, 224, 224, requires_grad=True, device=device)
  optimizer = torch.optim.Adam([reconstructed_image], lr=lr)

  for step in range(num_steps):

    if (step%100==0): print(f"Step {step}")
    reconstructed_features = loss_network(normalize(reconstructed_image))[layer_name]
    reconstructed_features = reconstructed_features.to(device)

    loss = mse_loss(gram(reconstructed_features), style_summary)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

  reconstructed_image = reconstructed_image.detach().clamp(0, 1)
  display(to_pil_image(reconstructed_image.squeeze(0)))

  return

In [ ]:
'''Version 4 of style reconstruction experiment: Visualizing style feature maps at each layer, with the style loss at layer l computed using the combined style loss
of the first l layers'''
def combined_reconstruction (target_image, layer_id, num_steps = 10000, lr = 0.1):
  selected = {}
  for index, layer_name in layers.items():
    if int(index) <= layer_id:
      selected[index] = layer_name

  vgg = vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features
  for param in vgg.parameters():
    param.requires_grad = False

  loss_network = create_feature_extractor(vgg, selected)
  loss_network = loss_network.to(device)

  target_features = loss_network(normalize(target_image))

  reconstructed_image = torch.randn_like(target_image, requires_grad=True)
  reconstructed_image = reconstructed_image.to(device)

  optimizer = torch.optim.Adam([reconstructed_image], lr=lr)

  for step in range(num_steps):
    if (step%100==0): print(f"Step {step}")
    reconstructed_features = loss_network(normalize(reconstructed_image))

    style_loss = calc_style_loss(reconstructed_features, target_features, selected)
    optimizer.zero_grad()
    style_loss.backward()
    optimizer.step()


  reconstructed_image = reconstructed_image.detach().clamp(0, 1)
  display(to_pil_image(reconstructed_image.squeeze(0)))


In [ ]:
'''Version 5 of style reconstruction experiment: I adjusted my code for Version 4 to visualize the reconstructed stylistic features that my second experiment
in the paper (calculating average Gram Matrices across each set of style-associated images for 3 styles being modelled) tries to capture. '''
def combined_reconstruction (style_id, layer_id, num_steps = 10000, lr = 0.1):
  selected = {}
  for index, layer_name in layers.items():
    if int(index) <= layer_id:
      selected[index] = layer_name

  vgg = vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features
  for param in vgg.parameters():
    param.requires_grad = False

  loss_network = create_feature_extractor(vgg, selected)
  loss_network = loss_network.to(device)

  style_features = precomputed_style_grams[style_id]

  reconstructed_image = torch.randn(1, 3, 224, 224, requires_grad=True, device=device)
  optimizer = torch.optim.Adam([reconstructed_image], lr=lr)

  for step in range(num_steps):
    if (step%1000==0): print(f"Step {step}")
    reconstructed_features = loss_network(normalize(reconstructed_image))
    style_loss = calc_style_loss_custom(reconstructed_features, style_features, selected)
    optimizer.zero_grad()
    style_loss.backward()
    optimizer.step()


  reconstructed_image = reconstructed_image.detach().clamp(0, 1)
  display(to_pil_image(reconstructed_image.squeeze(0)))